# 04 CNN Training

Stage 7 prepares PyTorch-ready tensors for deep learning, Stage 8 trains the first validation-monitored 1D CNN, and Stage 9 compares basic training choices for that same single-epoch CNN. This notebook starts with shape, leakage, and preprocessing checks, then runs the tiny overfit smoke test, the first training loop, and controlled validation-only Stage 9 experiments through reusable `src.train` utilities. The held-out test split is not evaluated here.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import torch

from src.data import (
    DEFAULT_EPOCH_INDEX_PATH,
    DEFAULT_PREPROCESSING_METADATA_PATH,
    DEFAULT_RAW_DATA_DIR,
    DreamtContextDataset,
    DreamtEpochDataset,
    DreamtSequenceDataset,
    check_epoch_split_leakage,
    fit_normalization_stats,
    load_preprocessing_metadata,
    save_preprocessing_metadata,
)
from src.train import (
    DEFAULT_STAGE8_OUTPUT_DIR,
    DEFAULT_STAGE9_OUTPUT_DIR,
    TrainConfig,
    build_stage9_screening_configs,
    run_tiny_overfit_test,
    run_stage9_experiments,
    train_model,
)

CHANNELS = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
BATCH_SIZE = 16
DEBUG_PARTICIPANTS = 3
EPOCHS = 1

raw_dir = repo_root / DEFAULT_RAW_DATA_DIR
epoch_index_path = repo_root / DEFAULT_EPOCH_INDEX_PATH
metadata_path = repo_root / DEFAULT_PREPROCESSING_METADATA_PATH
output_dir = repo_root / DEFAULT_STAGE8_OUTPUT_DIR
stage9_output_dir = repo_root / DEFAULT_STAGE9_OUTPUT_DIR
stage8_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=output_dir,
    channels=CHANNELS,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    max_train_participants=DEBUG_PARTICIPANTS,
    max_val_participants=DEBUG_PARTICIPANTS,
)

artifacts_available = raw_dir.exists() and epoch_index_path.exists()
artifacts_available


## Build Single-Epoch Datasets

In [ ]:
if artifacts_available:
    train_unscaled = DreamtEpochDataset(
        raw_dir=raw_dir,
        epoch_index=epoch_index_path,
        split="train",
        channels=CHANNELS,
        max_participants=DEBUG_PARTICIPANTS,
    )
    stats = fit_normalization_stats(train_unscaled)
    save_preprocessing_metadata(stats, metadata_path)
else:
    print("Skipping dataset construction because local raw files or epoch_index.csv are absent.")


In [ ]:
if artifacts_available:
    stats = load_preprocessing_metadata(metadata_path)
    train_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    val_ds = DreamtEpochDataset(raw_dir, epoch_index_path, split="validation", channels=CHANNELS, preprocessing_stats=stats, max_participants=DEBUG_PARTICIPANTS)
    check_epoch_split_leakage(train_ds.epoch_index)
    check_epoch_split_leakage(val_ds.epoch_index)
    loaders = {
        "train": torch.utils.data.DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True),
        "validation": torch.utils.data.DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False),
    }
    x_batch, y_batch = next(iter(loaders["train"]))
    print("train batch:", tuple(x_batch.shape), x_batch.dtype, tuple(y_batch.shape), y_batch.dtype)
    print("participants:", {"train": len(train_ds.participants), "validation": len(val_ds.participants)})
    print("metadata channels:", stats["channels"])


## Temporal Context And Sequence Shape Checks

In [ ]:
if artifacts_available:
    context_ds = DreamtContextDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, context_radius=2, max_participants=DEBUG_PARTICIPANTS)
    sequence_ds = DreamtSequenceDataset(raw_dir, epoch_index_path, split="train", channels=CHANNELS, preprocessing_stats=stats, sequence_length=5, label_mode="many_to_one", target_position="center", max_participants=DEBUG_PARTICIPANTS)
    if len(context_ds):
        x_context, y_context = context_ds[0]
        print("context item:", tuple(x_context.shape), y_context.item())
    if len(sequence_ds):
        x_sequence, y_sequence = sequence_ds[0]
        print("sequence item:", tuple(x_sequence.shape), y_sequence.item())


## Tiny Overfit Smoke Test

In [ ]:
if artifacts_available:
    overfit_history = run_tiny_overfit_test(train_ds, stage8_config)
    print("loss first/last:", round(overfit_history["loss"].iloc[0], 4), round(overfit_history["loss"].iloc[-1], 4))


## Single-Epoch CNN Training

In [ ]:
if artifacts_available:
    training_result = train_model(loaders["train"], loaders["validation"], stage8_config)
    display(training_result.history)
    print("best epoch:", training_result.best_epoch)
    print("outputs:", training_result.output_dir)


## Stage 9 Training-Choice Experiments

Stage 9 keeps the model family fixed to the single-epoch CNN and compares basic training choices using validation macro F1 as the primary selection metric. The default screening grid includes unweighted versus train-only class-weighted loss, learning rate, dropout including `0.0`, and weight decay. The test split remains untouched.

In [ ]:
stage9_base_config = TrainConfig(
    raw_dir=raw_dir,
    epoch_index_path=epoch_index_path,
    preprocessing_metadata_path=metadata_path,
    output_dir=stage9_output_dir,
    channels=CHANNELS,
    batch_size=32,
    epochs=40,
    patience=8,
    max_train_participants=DEBUG_PARTICIPANTS,
    max_val_participants=DEBUG_PARTICIPANTS,
)

stage9_screening_configs = build_stage9_screening_configs(
    base_config=stage9_base_config,
    output_dir=stage9_output_dir,
    learning_rates=(3e-4, 1e-3, 3e-3),
    dropouts=(0.0, 0.10, 0.25),
    weight_decays=(0.0, 1e-4, 1e-3),
    class_weighting_options=(False, True),
    batch_sizes=(32,),
)
len(stage9_screening_configs)


The full default grid has 54 runs. For a quick local dry run, slice `stage9_screening_configs` before calling `run_stage9_experiments`; for the main Stage 9 result, run the full list on the local machine/GPU.

In [ ]:
RUN_STAGE9_EXPERIMENTS = False

if artifacts_available and RUN_STAGE9_EXPERIMENTS:
    stage9_summary = run_stage9_experiments(
        stage9_screening_configs,
        output_dir=stage9_output_dir,
    )
    display(stage9_summary.sort_values("macro_f1", ascending=False).head(10))
    print("outputs:", stage9_output_dir)
else:
    print("Stage 9 experiments are configured but not run in this notebook execution.")
